# DiffusionNet Starter Notebook — Kaggle

**Đề tài:** Surface segmentation for 3D cultural-heritage fracture object using DiffusionNet  
**Tác giả:** 3D Fracture Segmentation Project

## Mục tiêu notebook này
Đây là "Hello World" cho project. Khi chạy xong toàn bộ, bạn sẽ:
1. Cài xong môi trường DiffusionNet trên Kaggle.
2. Load và visualize một mesh 3D.
3. Chạy forward pass DiffusionNet với nhãn giả, xác nhận pipeline hoạt động.

## ⚠️ Trước khi chạy — cấu hình Kaggle
Vào panel bên phải (Settings):
- **Accelerator**: `GPU T4 x2` (hoặc `GPU P100`)
- **Internet**: `On` (BẮT BUỘC — để cài thư viện từ GitHub)
- **Persistence**: `Files only`

Nếu không bật Internet, bước 1 sẽ fail.

## 1. Cài đặt môi trường

DiffusionNet cần: PyTorch (Kaggle có sẵn) + vài thư viện geometry processing.

In [ ]:
!pip install --quiet trimesh potpourri3d robust_laplacian plotly scikit-learn
!pip install --quiet matplotlib networkx
print('Install done ✓')

In [ ]:
# Kiểm tra GPU
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device sẽ dùng:', device)

## 2. Clone DiffusionNet và import

Nếu bạn đã upload DiffusionNet như một Kaggle Dataset thì bỏ qua clone, chỉ cần add dataset vào notebook và sửa `sys.path` cho đúng.

In [ ]:
import os, sys

DIFFNET_DIR = '/kaggle/working/diffusion-net'
if not os.path.exists(DIFFNET_DIR):
    !git clone https://github.com/nmwsharp/diffusion-net.git {DIFFNET_DIR}

sys.path.append(os.path.join(DIFFNET_DIR, 'src'))
import diffusion_net
print('DiffusionNet imported ✓')

## 3. Tải một mesh mẫu

Ta dùng Stanford Bunny (mesh kinh điển, ~5k vertices, nhẹ). Sau này thay bằng mesh cổ vật của bạn.

In [ ]:
import urllib.request, os

MESH_PATH = '/kaggle/working/bunny.obj'
# Stanford Bunny mirror — nếu link này hỏng, upload mesh bất kỳ rồi đổi MESH_PATH
URL = 'https://raw.githubusercontent.com/alecjacobson/common-3d-test-models/master/data/stanford-bunny.obj'
if not os.path.exists(MESH_PATH):
    urllib.request.urlretrieve(URL, MESH_PATH)
print('Mesh file:', MESH_PATH, '| size:', os.path.getsize(MESH_PATH), 'bytes')

In [ ]:
import trimesh
import numpy as np

mesh = trimesh.load(MESH_PATH, process=False)
verts = np.array(mesh.vertices, dtype=np.float32)
faces = np.array(mesh.faces, dtype=np.int64)

print('Số vertices:', verts.shape)   # (N, 3)
print('Số faces   :', faces.shape)   # (M, 3)
print('Bounding box:')
print('  min:', verts.min(axis=0))
print('  max:', verts.max(axis=0))

## 4. Visualize mesh (Plotly, chạy trong Kaggle được)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        color='lightgray', opacity=0.9,
    )
])
fig.update_layout(
    scene=dict(aspectmode='data'),
    title='Stanford Bunny — mesh gốc',
    height=500,
)
fig.show()

## 5. Tiền xử lý mesh (chuẩn hóa)

DiffusionNet thích mesh được normalize về bounding box đơn vị, tâm ở gốc tọa độ.

In [ ]:
def normalize_mesh(verts):
    verts = verts - verts.mean(axis=0)            # center
    scale = np.linalg.norm(verts, axis=1).max()   # scale radius = 1
    verts = verts / scale
    return verts.astype(np.float32)

verts = normalize_mesh(verts)
print('Sau normalize — max norm:', np.linalg.norm(verts, axis=1).max())
print('Tâm:', verts.mean(axis=0))

## 6. Tính operators cho DiffusionNet

DiffusionNet cần: `mass`, `L` (Laplacian), `evals`, `evecs`, `gradX`, `gradY`, `frames`.  
Thư viện có sẵn hàm `get_operators` tính tất cả và cache lại `.npz` để lần sau dùng lại nhanh.

In [ ]:
import torch
from diffusion_net.geometry import get_operators

verts_t = torch.tensor(verts, dtype=torch.float32)
faces_t = torch.tensor(faces, dtype=torch.long)

CACHE_DIR = '/kaggle/working/op_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

frames, mass, L, evals, evecs, gradX, gradY = get_operators(
    verts_t, faces_t,
    k_eig=128,        # số eigenvectors — 128 đủ dùng cho segmentation
    op_cache_dir=CACHE_DIR,
)
print('mass  :', mass.shape)
print('L     :', L.shape, '(sparse)')
print('evals :', evals.shape)
print('evecs :', evecs.shape)
print('gradX :', gradX.shape, '(sparse)')
print('gradY :', gradY.shape, '(sparse)')

## 7. Xây mô hình DiffusionNet

Cấu hình cho bài của bạn:
- Input: xyz (3 chiều) → có thể đổi sang HKS sau.
- Output: 2 class (fracture / original).
- 4 diffusion blocks, width 128.

In [ ]:
from diffusion_net.layers import DiffusionNet

model = DiffusionNet(
    C_in=3,              # input features = xyz
    C_out=2,             # nhãn: 0 = original, 1 = fracture
    C_width=128,
    N_block=4,
    last_activation=None,  # để raw logits, dùng với CrossEntropyLoss
    outputs_at='vertices', # segmentation per-vertex
    dropout=True,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'DiffusionNet: {n_params/1e6:.2f}M params')

## 8. Forward pass với nhãn giả

Sinh ngẫu nhiên nhãn (0/1) cho mỗi vertex để xác nhận model chạy, loss tính được, gradient lan về được.

In [ ]:
import torch.nn.functional as F

# Chuyển tensors lên GPU
verts_gpu = verts_t.to(device)
faces_gpu = faces_t.to(device)
mass_gpu  = mass.to(device)
L_gpu     = L.to(device)
evals_gpu = evals.to(device)
evecs_gpu = evecs.to(device)
gradX_gpu = gradX.to(device)
gradY_gpu = gradY.to(device)

# Input features = xyz (3 chiều)
x_in = verts_gpu

# Forward
model.train()
logits = model(
    x_in=x_in,
    mass=mass_gpu,
    L=L_gpu,
    evals=evals_gpu,
    evecs=evecs_gpu,
    gradX=gradX_gpu,
    gradY=gradY_gpu,
    faces=faces_gpu,
)
print('Logits shape:', logits.shape, '  (N_vertices, 2)')

# Fake labels — random 0/1
torch.manual_seed(42)
fake_labels = torch.randint(0, 2, (verts.shape[0],), device=device)

loss = F.cross_entropy(logits, fake_labels)
print('Loss (fake labels):', loss.item())

loss.backward()
print('Backward ✓ — gradient đã lan được qua toàn bộ network')

## 9. Visualize prediction (để cảm nhận pipeline hoàn chỉnh)

Model chưa train nên prediction sẽ gần như ngẫu nhiên — nhưng ta sẽ vẽ vertices theo class để bạn thấy kết quả sau train sẽ trông thế nào.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(
        x_in=x_in, mass=mass_gpu, L=L_gpu,
        evals=evals_gpu, evecs=evecs_gpu,
        gradX=gradX_gpu, gradY=gradY_gpu, faces=faces_gpu,
    )
    pred = logits.argmax(dim=-1).cpu().numpy()

print('Vertex class distribution:', np.bincount(pred))

# Màu hóa: 0=xám, 1=đỏ
vertex_colors = np.where(pred[:, None] == 1,
                         np.array([[220, 50, 50]]),
                         np.array([[200, 200, 200]]))

fig = go.Figure(data=[
    go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        vertexcolor=vertex_colors, opacity=1.0,
    )
])
fig.update_layout(
    scene=dict(aspectmode='data'),
    title='Prediction thử (model chưa train — random)',
    height=500,
)
fig.show()

## Đã chạy tới đây nghĩa là gì?

Bạn đã dựng xong toàn bộ khung:
- ✅ Môi trường Kaggle + GPU hoạt động.
- ✅ DiffusionNet cài và import được.
- ✅ Load, normalize, visualize mesh 3D.
- ✅ Tính operators (mass, Laplacian, eigenvectors...) cho DiffusionNet.
- ✅ Model forward + backward thành công.
- ✅ Visualize prediction trên mesh.

## Bước tiếp theo (Tuần 2 trở đi)

1. **Thay Stanford Bunny bằng dataset thật**: tải Breaking Bad Dataset hoặc tự tạo synthetic từ Thingi10K.
2. **Viết Dataset class PyTorch**: load nhiều mesh, precompute operators 1 lần, cache lại.
3. **Viết training loop**: chia train/val, loss với class weighting cho imbalance.
4. **Tính metric đúng**: Accuracy, Precision, Recall, F1, IoU per-class.
5. **Thử input features khác**: thay `xyz` bằng HKS (16-dim) để so sánh.

Chúc bạn chạy thành công! Nếu cell nào lỗi, copy error ra báo lại để mình fix.